# 第 15 章 不完全競争とゲーム理論: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/15-imperfect-competition-and-game-theory.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 需要 Q = a - bP の切片 a
demand_slope = 2        # 需要の傾き b
marginal_cost = 10      # 限界費用（一定）
firms = 2               # クールノー競争の企業数
supply_intercept = 0    # 外部不経済: 私的な供給 Q = c + dP の切片 c
supply_slope = 3        # 私的な供給の傾き d
external_cost = 5       # 外部費用（1 単位あたり）
both_cooperate = 3      # ゲーム: 2 人とも協力したときの、利得
temptation = 5          # 自分だけ裏切ったときの、利得
sucker = 0              # 自分だけ協力したときの、利得
both_defect = 1         # 2 人とも裏切ったときの、利得

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve().parent != _root / "lib":
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 独占

In [ ]:
from lib.chapter15.monopoly import competitive_quantity, lerner_index, monopoly_outcome

outcome = monopoly_outcome(demand_intercept, demand_slope, marginal_cost)
print(f"独占: 数量 {outcome.quantity:.2f}、価格 {outcome.price:.2f}、"
      f"利潤 {outcome.profit:.2f}")
print(f"消費者余剰 {outcome.consumer_surplus:.2f}、"
      f"死荷重 {outcome.deadweight_loss:.2f}")
print(f"競争の数量: "
      f"{competitive_quantity(demand_intercept, demand_slope, marginal_cost):.2f}")
print(f"ラーナー指数: {lerner_index(outcome.price, marginal_cost):.3f}")

In [ ]:
from lib.chapter15.plot import plot_monopoly

plot_monopoly(demand_intercept, demand_slope, marginal_cost)

## 寡占（クールノー競争）

In [ ]:
from lib.chapter15.oligopoly import cournot_by_iteration, cournot_symmetric

q1, q2 = cournot_by_iteration(demand_intercept, demand_slope, marginal_cost)
print(f"2 社の反復で求めた均衡: {q1:.2f}、{q2:.2f}")
quantity, price, profit = cournot_symmetric(
    firms, demand_intercept, demand_slope, marginal_cost
)
print(f"{firms} 社: 各社の数量 {quantity:.2f}、価格 {price:.2f}、"
      f"各社の利潤 {profit:.2f}")

In [ ]:
from lib.chapter15.plot import plot_cournot

plot_cournot(demand_intercept, demand_slope, marginal_cost)

## 外部不経済とピグー税

In [ ]:
from lib.chapter15.externality import (
    externality_deadweight_loss,
    social_optimum_quantity,
)

market = (demand_intercept, demand_slope, supply_intercept, supply_slope)
print(f"社会的に最適な数量: {social_optimum_quantity(*market, external_cost):.2f}")
print(f"死荷重: {externality_deadweight_loss(*market, external_cost):.2f}")

In [ ]:
from lib.chapter15.plot import plot_externality

plot_externality(*market, external_cost)

## ゲーム理論

In [ ]:
from lib.chapter15.game import pure_nash_equilibria

payoffs = {
    ("協力", "協力"): (both_cooperate, both_cooperate),
    ("協力", "裏切り"): (sucker, temptation),
    ("裏切り", "協力"): (temptation, sucker),
    ("裏切り", "裏切り"): (both_defect, both_defect),
}
print("ナッシュ均衡:", pure_nash_equilibria(payoffs))

## 試してみよう

- `marginal_cost` を 20 にすると、独占の数量は 40 から 30 に、価格は 30 から 35 に、
  利潤は 800 から 450 になります。`demand_slope` を 4 にすると、数量は 30、価格は
  17.50 です（`marginal_cost` は 10 のまま）
- `firms` を 1 にすると、独占と同じ（数量 40、価格 30、利潤 800）です。10 にすると、
  各社の数量は 7.27、価格は 13.64、各社の利潤は 26.45 に下がり、競争に近づきます
  （2 社なら、26.67、23.33、355.56）
- `external_cost` を 10 にすると、最適な数量は 54 から 48 に、死荷重は 15 から 60 に
  なります。0 にすると、死荷重は 0 です
- `both_cooperate` を 6 にすると、ナッシュ均衡が 2 つ（協力・協力と、裏切り・裏切り）
  になります（3 のときは、裏切り・裏切りだけ）